In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../../"))

import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load data
df_titanic = pd.read_csv('../../../../data/raw/structured/titanic.csv')
print("Data loaded successfully!")
df_titanic.head()

In [ ]:
# Data preprocessing
df_titanic.drop(["PassengerId", "Name", "Ticket", "Cabin", "Fare", "Parch"], axis=1, inplace=True, errors='ignore')
df_titanic = df_titanic.dropna(subset=['Embarked'])

mean_age = df_titanic['Age'].mean()
df_titanic['Age'].fillna(mean_age, inplace=True)

# Convert categorical to numeric
df_titanic["Sex"] = df_titanic["Sex"].map({"male": 0, "female": 1})
df_titanic["Embarked"] = df_titanic["Embarked"].map({"S": 0, "C": 1, "Q": 2})
df_titanic.fillna(df_titanic.median(numeric_only=True), inplace=True)

y = df_titanic["Survived"].values
X = df_titanic.drop(["Survived"], axis=1)

print("Preprocessing complete!")
X.head()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=9
)

# Train model
clf = GaussianNB()
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluate
print(f"Accuracy: {clf.score(X_test, y_test):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Cross-validation
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []
accuracy_scores = []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    clf = GaussianNB()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    accuracy_scores.append(accuracy_score(y_test, y_pred))
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"Mean Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Std Accuracy: {np.std(accuracy_scores):.4f}")
print("\nOverall Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=['Not Survived', 'Survived']))